# [Real Time Speech Recognition](https://www.gradio.app/guides/real-time-speech-recognition)

`pip install torch transformers torchaudio`

or make sure they're all in `requirements.txt` and run `pip install -r requirements`

In [ ]:
from transformers import pipeline

p = pipeline("automatic-speech-recognition", model="openai/whisper-base.en")

## [Full-Context ASR Demo with Transformers](https://www.gradio.app/guides/real-time-speech-recognition#2-create-a-full-context-asr-demo-with-transformers)

In [ ]:
import gradio as gr
from transformers import pipeline
import numpy as np

transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-base.en")

def transcribe(audio):
    sr, y = audio
    
    # Convert to mono if stereo
    if y.ndim > 1:
        y = y.mean(axis=1)
        
    y = y.astype(np.float32)
    y /= np.max(np.abs(y))

    return transcriber({"sampling_rate": sr, "raw": y})["text"]  

demo = gr.Interface(
    transcribe,
    gr.Audio(sources="microphone"),
    "text",
)

demo.launch()

## [Streaming ASR Demo with Transformers](https://www.gradio.app/guides/real-time-speech-recognition#3-create-a-streaming-asr-demo-with-transformers)

In [ ]:
import gradio as gr
from transformers import pipeline
import numpy as np

transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-base.en")

def transcribe(stream, new_chunk):
    sr, y = new_chunk
    
    # Convert to mono if stereo
    if y.ndim > 1:
        y = y.mean(axis=1)
        
    y = y.astype(np.float32)
    y /= np.max(np.abs(y))

    if stream is not None:
        stream = np.concatenate([stream, y])
    else:
        stream = y
    return stream, transcriber({"sampling_rate": sr, "raw": stream})["text"]  

demo = gr.Interface(
    transcribe,
    ["state", gr.Audio(sources=["microphone"], streaming=True)],
    ["state", "text"],
    live=True,
)

demo.launch()